# VisionGuard: AI vs Real Image Classification Training Pipeline

**Student:** Shayan Ahmed  
**Roll Number:** 23L-0801  
**Project:** VisionGuard (`23L-0801_A2_Ops`)  
**Repository:** [github.com/shayanO-O/23L-0801_A2_Ops](https://github.com/shayanO-O/23L-0801_A2_Ops)  
**DagsHub Project:** [dagshub.com/shayanO-O/23L-0801_A2_Ops](https://dagshub.com/shayanO-O/23L-0801_A2_Ops)  

### Instructions Before Running:
1. In the menu bar, navigate to **Runtime > Change runtime type** and select **T4 GPU**.
2. Open the **Secrets** panel (key icon on the left sidebar) and add the following two secrets:
   - `DAGSHUB_USER`: Your DagsHub username (`shayanO-O`)
   - `DAGSHUB_TOKEN`: Your DagsHub Personal Access Token
   - Ensure the toggle **Notebook access** is switched **ON** for both secrets.
3. Click **Runtime > Run all**.
4. Keep this browser tab open and your laptop awake until all cells complete.

In [ ]:
# Cell 1: Environment & Platform Versions Check
import sys
import tensorflow as tf
import keras
import mlflow

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"MLflow Version: {mlflow.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {bool(gpus)}")
if gpus:
    print(f"GPU Device: {gpus[0]}")

In [ ]:
# Cell 2: Clone Public GitHub Repository
import os
import shutil

REPO_URL = "https://github.com/shayanO-O/23L-0801_A2_Ops.git"
WORKSPACE_DIR = "/content/23L-0801_A2_Ops"

if os.path.exists(WORKSPACE_DIR):
    shutil.rmtree(WORKSPACE_DIR)

!git clone {REPO_URL} {WORKSPACE_DIR}
%cd {WORKSPACE_DIR}
!git log -1 --oneline

In [ ]:
# Cell 3: Install Pinned Training Dependencies
!pip install --upgrade pip
!pip install -r requirements-train.txt

# Re-verify Versions Match Exact Pins
import tensorflow as tf, keras, mlflow, dvc
print(f"Verified TF: {tf.__version__}, Keras: {keras.__version__}, MLflow: {mlflow.__version__}")

In [ ]:
# Cell 4: Pull Dataset From DagsHub DVC Storage Using Secrets
from google.colab import userdata
import subprocess

dagshub_user = userdata.get('DAGSHUB_USER')
dagshub_token = userdata.get('DAGSHUB_TOKEN')

# Configure Remote DVC Credentials Safely In Local Git-Ignored Config
subprocess.run(['dvc', 'remote', 'modify', 'origin', '--local', 'auth', 'basic'], check=True)
subprocess.run(['dvc', 'remote', 'modify', 'origin', '--local', 'user', dagshub_user], check=True)
subprocess.run(['dvc', 'remote', 'modify', 'origin', '--local', 'password', dagshub_token], check=True)

# Pull Dataset Content
!dvc pull -r origin
!ls -la data/raw

In [ ]:
# Cell 5: Execute Model Training Pipeline (R0 to R6, Tuner, Finals)
from google.colab import userdata
import os

dagshub_user = userdata.get('DAGSHUB_USER')
dagshub_token = userdata.get('DAGSHUB_TOKEN')

# Pass MLflow Credentials Via Environment Variables
os.environ['MLFLOW_TRACKING_URI'] = f"https://dagshub.com/{dagshub_user}/23L-0801_A2_Ops.mlflow"
os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_user
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
os.environ['MLFLOW_EXPERIMENT_NAME'] = '23L-0801_A2_Ops'

# Launch Full Training Sequence
!python src/experiments.py --stage all

In [ ]:
# Cell 6: Confirmation And Link To Remote Experiments
print("=================================================================")
print("  VisionGuard Colab Training Run Completed Successfully!        ")
print(f"  View runs on DagsHub: https://dagshub.com/{dagshub_user}/23L-0801_A2_Ops/experiments")
print("=================================================================")